# AutoSort Training Pipeline

Main steps:
1. Threshold detection
2. Training data preparation
3. Model training


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils_clique import (
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques,
    CliqueInfo,
    plot_cliques,
    compute_valid_channels
)


In [2]:
# Load 384-channel data using MEArec format
recording_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5"
spike_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/neuron_inf.pkl"

# Load recording and sorting using MEArec
recording, sorting = se.read_mearec(recording_path)

# Get probe from recording
probe = recording.get_probe()
if probe is None:
    raise ValueError("Recording does not have probe information")

# Build cliques from probe
cliques = build_sliding_cliques(
    probe,
    clique_size=49,
    min_size=25,
    min_overlap=18,
    target_groups=12,
)

# Plot cliques visualization
output_dir = Path("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/")
output_dir.mkdir(parents=True, exist_ok=True)
plot_cliques(probe, cliques, output_pdf_path=str(output_dir / "cliques_visualization.pdf"))

# Save clique information for evaluation
clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 18,
        'target_groups': 12,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
    'recording_path': recording_path,  # Recording path for reference
}

clique_info_path = output_dir / "clique_info.pkl"
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 12 cliques (target 12)
       Clique 00: channels 192-12 (49 channels)
       Clique 01: channels 103-115 (49 channels)
       Clique 02: channels 303-123 (49 channels)
       Clique 03: channels 23-227 (49 channels)
       Clique 04: channels 31-43 (49 channels)
       Clique 05: channels 326-338 (49 channels)
       Clique 06: channels 334-154 (49 channels)
       Clique 07: channels 54-258 (49 channels)
       Clique 08: channels 254-74 (49 channels)
       Clique 09: channels 165-177 (49 channels)
       Clique 10: channels 173-185 (49 channels)
       Clique 11: channels 371-383 (49 channels)

Clique可视化PDF已保存至: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/cliques_visualization.pdf


In [3]:
# Preprocess recording
# recording_f = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
# recording_f = spre.common_reference(recording_f, reference="global", operator="median")
recording_f = recording.rename_channels(range(384))
# Load GT data
spike_inf = None
neuron_inf = None

if Path(spike_inf_path).exists():
    spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
else:
    raise ValueError(f"spike_inf not found at {spike_inf_path}")

if Path(neuron_inf_path).exists():
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf = pickle.load(f)
else:
    raise ValueError(f"neuron_inf.pkl not found at {neuron_inf_path}")

## Step 1: Threshold Detection + Training Data Preparation


In [4]:
# Set parameters
from typing import Any


base_save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/"
duration_seconds = 200  # Processing duration (seconds)

# Check if required data is loaded
if spike_inf is None:
    raise ValueError("spike_inf is required but not loaded")
if neuron_inf is None:
    raise ValueError("neuron_inf is required but not loaded")

# Detection parameters (consistent with AutoSort default values)
detection_params = {
    'thr_min': 2.7,
    'thr_max': 15,
    'distance': 4,
    'ch_max_simul_firing': 8,
    'wlen': 6,
    'prominence': 10,
}

# Waveform window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Process each clique
probe_df = probe.to_dataframe()
all_train_data_dirs = {}

for clique_id, clique in enumerate(cliques):
    # Create clique-specific save directory
    save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Get clique channels (probe channel indices, 0-based)
    clique_channels = set(clique.device_channel_indices)
    
    # Create recording_clique by selecting clique channels from full recording
    # Since channel_ids are now 0-based integers, we can directly use them
    clique_channel_ids = list(clique_channels)  # Direct use of 0-based integer indices
    recording_clique = recording_f.select_channels(channel_ids=clique_channel_ids)
    
    # Filter neurons based on channel_id: only keep neurons whose all channel_id are in clique
    neuron_ids_in_clique = []
    for idx, row in neuron_inf.iterrows():
        neuron_channel_ids = row.get('channel_id', [])
        # Handle different formats: list, string representation of list, etc.
        if isinstance(neuron_channel_ids, str):
            import ast
            try:
                neuron_channel_ids = ast.literal_eval(neuron_channel_ids)
            except:
                neuron_channel_ids = []
        elif not isinstance(neuron_channel_ids, (list, tuple, np.ndarray)):
            neuron_channel_ids = []
        
        # Convert to set for easy comparison
        neuron_channel_set = set[Any](neuron_channel_ids)
        
        # Check if all channels of this neuron are in the clique
        if len(neuron_channel_set) > 0 and neuron_channel_set.issubset(clique_channels):
            neuron_ids_in_clique.append(row['Neuron'])
    
    # Filter neuron_inf
    neuron_inf_clique = neuron_inf[neuron_inf['Neuron'].isin(neuron_ids_in_clique)].copy()
    
    # cluster_ids_in_clique should contain neuron IDs (not cluster IDs)
    # This is what spike_inf uses to match spikes to neurons
    cluster_ids_in_clique = neuron_ids_in_clique.copy()
    
    # Filter spike_inf to clusters/neurons within clique
    # Check which column name spike_inf uses ('neuron' or 'cluster')
    if 'neuron' in spike_inf.columns:
        spike_inf_clique = spike_inf[spike_inf['neuron'].isin(cluster_ids_in_clique)].copy()
    elif 'cluster' in spike_inf.columns:
        spike_inf_clique = spike_inf[spike_inf['cluster'].isin(cluster_ids_in_clique)].copy()
    else:
        raise ValueError(f"spike_inf must have either 'neuron' or 'cluster' column")
    
    if len(spike_inf_clique) == 0:
        continue
    
    # Compute valid_channels: only channels that have GT neurons
    print(f"\n=== Computing valid channels for Clique {clique_id} ===")
    try:
        valid_channels = compute_valid_channels(
            recording_clique=recording_clique,
            neuron_inf_clique=neuron_inf_clique,
        )
        print(f"Valid channels (with GT neurons): {len(valid_channels) if valid_channels is not None else 'all'} channels")
        if valid_channels is not None:
            print(f"  Valid channel indices: {valid_channels[:10]}{'...' if len(valid_channels) > 10 else ''}")
    except Exception as e:
        print(f"Warning: Failed to compute valid channels for Clique {clique_id}: {e}")
        print(f"Using all channels")
        valid_channels = None
    
    # Prepare training data for this clique
    # Use recording_clique for clique-level detection
    # valid_channels: only detect on channels that have GT neurons
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,  # Use clique recording for clique-level detection
        spike_inf=spike_inf_clique,
        neuron_inf=neuron_inf_clique,
        save_dir=str(save_dir),
        duration_seconds=duration_seconds,
        valid_channels=valid_channels,  # Only channels with GT neurons
        **detection_params,
        **window_params
    )
    
    all_train_data_dirs[clique_id] = train_data_dir



=== Computing valid channels for Clique 0 ===
Valid channels (with GT neurons): 9 channels
  Valid channel indices: [0, 6, 9, 16, 19, 21, 29, 30, 41]
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 49
Recording total length: 36000000 samples (3600.00 seconds)
Will process first 2000000 samples (200.00 seconds)
Data shape: (2000000, 49) (clique channels)
Building detect_array...
Number of detected spikes: 418369

### 2. Load Ground Truth and Match
Building gt_array...
GT spike count: 17446
---spike detection rate: 0.9446
Number of matched spikes: 16480
Number of unmatched spikes: 401889

### 3. Extract Waveforms


Extracting waveforms: 100%|██████████| 30/30 [00:13<00:00,  2.16it/s]


Waveform extraction completed!
waveform shape: (418358, 49, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_00/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_00/train_data
Data statistics:
  - Total spike count: 418358
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 9
  - Noise spike count: 401878
  - Valid spike count: 16480

=== Computing valid channels for Clique 1 ===
Valid channels (with GT neurons): 11 channels
  Valid channel indices: [1, 4, 7, 8, 9, 12, 15, 16, 39, 43]..

Extracting waveforms: 100%|██████████| 30/30 [00:14<00:00,  2.07it/s]


Waveform extraction completed!
waveform shape: (415585, 49, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_01/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_01/train_data
Data statistics:
  - Total spike count: 415585
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 12
  - Noise spike count: 398218
  - Valid spike count: 17367

=== Computing valid channels for Clique 2 ===
Valid channels (with GT neurons): 12 channels
  Valid channel indices: [1, 3, 5, 9, 10, 20, 22, 34, 38, 40]

...

Waveform extraction completed!
waveform shape: (415845, 49, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_09/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_09/train_data
Data statistics:
  - Total spike count: 415845
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 13
  - Noise spike count: 400063
  - Valid spike count: 15782

=== Computing valid channels for Clique 10 ===
Valid channels (with GT neurons): 12 channels
  Valid channel indices: [3, 4, 6, 9, 13, 30, 31, 33, 39, 40

Extracting waveforms: 100%|██████████| 30/30 [00:14<00:00,  2.06it/s]


Waveform extraction completed!
waveform shape: (424007, 49, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_10/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_10/train_data
Data statistics:
  - Total spike count: 424007
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 12
  - Noise spike count: 411516
  - Valid spike count: 12491

=== Computing valid channels for Clique 11 ===
Valid channels (with GT neurons): 9 channels
  Valid channel indices: [3, 17, 26, 29, 34, 35, 42, 45, 47]


Extracting waveforms: 100%|██████████| 30/30 [00:14<00:00,  2.08it/s]


Waveform extraction completed!
waveform shape: (417061, 49, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_11/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_11/train_data
Data statistics:
  - Total spike count: 417061
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 9
  - Noise spike count: 407630
  - Valid spike count: 9431


## Step 2: Model Training


In [5]:
# Set training parameters
training_params = {
    'epochs': 20,
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
    'early_stopping': True,  # Enable early stopping
    'patience': 5,  # Stop if accuracy doesn't improve for 5 consecutive epochs
    'min_delta': 0.0,  # Minimum change
}

# Repeat training 5 times per clique
n_runs = 5

# Train models for each clique
all_models_dict = {}
all_logs_dict = {}

for clique_id, clique in enumerate(cliques):
    if clique_id not in all_train_data_dirs:
        print(f"Skipping clique {clique_id} (no training data)")
        continue
    
    print(f"\n{'='*80}")
    print(f"Training models for Clique {clique_id:02d}")
    print(f"{'='*80}")
    
    train_data_dir = all_train_data_dirs[clique_id]
    base_model_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save"
    
    # Get number of channels for this clique
    n_channels = len(clique.device_channel_indices)
    
    all_models = []
    all_logs = []
    
    for run_id in range(1, n_runs + 1):
        print(f"\n{'-'*60}")
        print(f"Clique {clique_id:02d} - Training run {run_id}/{n_runs}")
        print(f"{'-'*60}")
        
        # Create independent save directory for each training run
        model_save_dir = base_model_save_dir / f"run_{run_id}"
        model_save_dir.mkdir(parents=True, exist_ok=True)
        
        # Train model
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=str(model_save_dir),
            n_channels=n_channels,
            **training_params
        )
        
        all_models.append(autosort_model)
        all_logs.append(training_log)
        
        print(f"Clique {clique_id:02d} - Run {run_id} completed!")
        print(f"Model save directory: {model_save_dir}")
    
    all_models_dict[clique_id] = all_models
    all_logs_dict[clique_id] = all_logs
    print(f"\nClique {clique_id:02d} - All {n_runs} training runs completed!")

print(f"\n{'='*80}")
print(f"All cliques training completed!")
print(f"{'='*80}")



Training models for Clique 00

------------------------------------------------------------
Clique 00 - Training run 1/5
------------------------------------------------------------
Using device: cuda
Create dataset...
Dataset loaded:
  - Total samples: 418358
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 9
  - Noise samples: 401878.0
  - Non-noise samples: 16480.0
Model parameters:
  - Number of channels: 49
  - Window length: 30
  - Number of units: 9
  - Input dimension: 1500
Unit ID list saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_00/model_save/run_1/keep_id.pkl

Dataset split:
  - Training set: 334686 samples
  - Validation set: 83672 samples

Starting training (total 20 epochs)...
Early stopping enabled: patience=5, min_delta=0.0
epoch : 1/20


Training: 100%|██████████| 654/654 [00:05<00:00, 126.51it/s]


epoch : 1/20, detection loss = 243.176718, classification loss = 444.663419


Validation: 100%|██████████| 164/164 [00:00<00:00, 215.59it/s]


epoch : 1/20, val detection loss = 160.973277, classification loss = 188.273963
Validation Loss Decreased(inf--->349.247241)
Validation Accuracy Decreased(inf--->0.926403) 	 Saving The Model (Best Acc Epoch)
epoch : 2/20


...

epoch : 5/20, val detection loss = 125.517653, classification loss = 27.127543
epoch : 6/20


Training: 100%|██████████| 652/652 [00:04<00:00, 133.95it/s]


epoch : 6/20, detection loss = 42.447504, classification loss = 42.786930


Validation: 100%|██████████| 163/163 [00:00<00:00, 212.81it/s]


epoch : 6/20, val detection loss = 165.526151, classification loss = 25.486568

Early stopping triggered: 5 consecutive epochs without improvement
Best accuracy: 0.936677 (Epoch 1)

Training completed! Training log saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_11/model_save/run_5/training_log.csv
Best validation accuracy: 0.936677 (Epoch 1)
Clique 11 - Run 5 completed!
Model save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_11/model_save/run_5

Clique 11 - All 5 training runs completed!

All cliques training completed!
